# Feature Engineering & Proxy Target Variable Creation

This notebook engineers features from raw transaction data and creates a proxy default target variable using RFM-based customer segmentation.

**Objectives:**
1. Calculate RFM (Recency, Frequency, Monetary) metrics
2. Engineer aggregate and temporal features
3. Create proxy target using K-Means clustering
4. Build reproducible sklearn Pipeline for feature transformation

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Load raw data
df = pd.read_csv('../data/data.csv')
print(f"Original dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

## 2. Data Preparation and DateTime Parsing

In [ ]:
# Parse datetime column
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'], errors='coerce')

# Define snapshot date for RFM calculation (last date in dataset)
snapshot_date = df['TransactionStartTime'].max()
print(f"Snapshot date: {snapshot_date}")
print(f"Data range: {df['TransactionStartTime'].min()} to {snapshot_date}")

# Create a copy for feature engineering
df_features = df.copy()

## 3. Engineer RFM Features

In [ ]:
# Group by customer and calculate RFM metrics
rfm_data = []

for customer_id in df_features['CustomerId'].unique():
    customer_data = df_features[df_features['CustomerId'] == customer_id]
    
    # Recency: Days since last transaction
    last_transaction = customer_data['TransactionStartTime'].max()
    recency = (snapshot_date - last_transaction).days
    
    # Frequency: Number of transactions
    frequency = len(customer_data)
    
    # Monetary: Total transaction value
    monetary = customer_data['Value'].sum()
    
    rfm_data.append({
        'CustomerId': customer_id,
        'Recency': recency,
        'Frequency': frequency,
        'Monetary': monetary
    })

rfm_df = pd.DataFrame(rfm_data)
print(f"RFM features shape: {rfm_df.shape}")
print(f"\nRFM Summary Statistics:")
print(rfm_df.describe())

## 4. Create Proxy Target with K-Means Clustering

In [ ]:
# Prepare RFM features for clustering
rfm_features = rfm_df[['Recency', 'Frequency', 'Monetary']].copy()

# Scale RFM features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

# Apply K-Means clustering with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm_df['Cluster'] = kmeans.fit_predict(rfm_scaled)

print(f"Cluster distribution:")
print(rfm_df['Cluster'].value_counts().sort_index())

# Analyze cluster characteristics
print(f"\nCluster Characteristics:")
cluster_analysis = rfm_df.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()
print(cluster_analysis)

In [ ]:
# Identify high-risk cluster (low frequency, low monetary)
# High-risk customers are disengaged (low activity)
cluster_scores = rfm_df.groupby('Cluster').agg({
    'Frequency': 'mean',
    'Monetary': 'mean',
    'Recency': 'mean'
})

# Calculate risk score (lower frequency and monetary = higher risk)
cluster_scores['Risk_Score'] = (1 / (cluster_scores['Frequency'] + 1)) * (1 / (cluster_scores['Monetary'] + 1))
print(f"\nCluster Risk Analysis:")
print(cluster_scores)

# Identify high-risk cluster
high_risk_cluster = cluster_scores['Risk_Score'].idxmax()
print(f"\nHigh-risk cluster identified: {high_risk_cluster}")

# Create binary target variable
rfm_df['is_high_risk'] = (rfm_df['Cluster'] == high_risk_cluster).astype(int)
print(f"\nTarget variable distribution:")
print(rfm_df['is_high_risk'].value_counts())

In [ ]:
# Visualize clusters
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Recency vs Frequency
scatter1 = axes[0].scatter(rfm_df['Recency'], rfm_df['Frequency'], 
                           c=rfm_df['Cluster'], cmap='viridis', alpha=0.6)
axes[0].set_xlabel('Recency (days)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Recency vs Frequency')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Frequency vs Monetary
scatter2 = axes[1].scatter(rfm_df['Frequency'], rfm_df['Monetary'], 
                           c=rfm_df['Cluster'], cmap='viridis', alpha=0.6)
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Monetary Value')
axes[1].set_title('Frequency vs Monetary')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

# Target distribution
target_counts = rfm_df['is_high_risk'].value_counts()
axes[2].bar(['Low Risk', 'High Risk'], target_counts.values)
axes[2].set_ylabel('Count')
axes[2].set_title('Target Variable Distribution')

plt.tight_layout()
plt.show()

## 5. Engineer Additional Features

In [ ]:
# Extract temporal features
df_features['TransactionHour'] = df_features['TransactionStartTime'].dt.hour
df_features['TransactionDay'] = df_features['TransactionStartTime'].dt.day
df_features['TransactionMonth'] = df_features['TransactionStartTime'].dt.month
df_features['TransactionYear'] = df_features['TransactionStartTime'].dt.year
df_features['DayOfWeek'] = df_features['TransactionStartTime'].dt.dayofweek

# Create aggregate features per customer
aggregate_features = df_features.groupby('CustomerId').agg({
    'Value': ['sum', 'mean', 'std', 'min', 'max', 'count'],
    'Amount': ['sum', 'mean', 'std'],
    'TransactionHour': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    'ProductCategory': 'nunique',
    'ChannelId': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    'FraudResult': ['sum', 'mean']
}).reset_index()

# Flatten column names
aggregate_features.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                               for col in aggregate_features.columns.values]

print(f"Aggregate features shape: {aggregate_features.shape}")
print(f"\nAggregate features columns:")
print(aggregate_features.columns.tolist())

In [ ]:
# Merge all features with target
customer_level_df = aggregate_features.merge(rfm_df[['CustomerId', 'is_high_risk']], 
                                               on='CustomerId', how='inner')

print(f"Final customer-level dataset shape: {customer_level_df.shape}")
print(f"\nDataset preview:")
print(customer_level_df.head())
print(f"\nData types:")
print(customer_level_df.dtypes)

## 6. Build Reproducible sklearn Pipeline

In [ ]:
# Identify feature types
numerical_features = customer_level_df.select_dtypes(include=[np.number]).columns.tolist()
numerical_features.remove('CustomerId')  # Remove ID column
if 'is_high_risk' in numerical_features:
    numerical_features.remove('is_high_risk')  # Remove target

categorical_features = customer_level_df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical features ({len(numerical_features)}): {numerical_features[:5]}...")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

In [ ]:
# Define preprocessing pipelines
# Numerical pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combined preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ])

print("Preprocessing pipeline created successfully!")
print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

In [ ]:
# Fit and transform data
X = customer_level_df.drop(['CustomerId', 'is_high_risk'], axis=1)
y = customer_level_df['is_high_risk']

X_transformed = preprocessor.fit_transform(X)
print(f"Transformed feature matrix shape: {X_transformed.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nClass balance: {(y.sum() / len(y) * 100):.2f}% high-risk customers")

In [ ]:
# Save processed data
processed_data = pd.DataFrame(X_transformed)
processed_data['is_high_risk'] = y.values

# Save with customer IDs for reference
processed_data_with_id = pd.DataFrame(X_transformed)
processed_data_with_id['CustomerId'] = customer_level_df['CustomerId'].values
processed_data_with_id['is_high_risk'] = y.values

processed_data.to_csv('../data/processed/processed_features.csv', index=False)
processed_data_with_id.to_csv('../data/processed/processed_features_with_id.csv', index=False)

print("Processed data saved successfully!")
print(f"Location: ../data/processed/processed_features.csv")

## Summary

✅ **Completed Tasks:**
1. Calculated RFM metrics for customer segmentation
2. Applied K-Means clustering to identify high-risk customers
3. Created binary proxy target variable `is_high_risk`
4. Engineered temporal and aggregate features
5. Built reproducible sklearn ColumnTransformer pipeline
6. Saved processed data for model training

**Key Insights:**
- High-risk customers identified as disengaged segments with low frequency and monetary value
- Target variable ready for supervised learning models
- Pipeline ensures reproducibility and handles missing values
- All transformation steps are automatable for production inference